# Full 21-Day VLM 行为画像分析

针对 `output/analysis/full_21d_vlm/` 的分析结果做探索：

1. **全局画像** — `behavior_profile`（兴趣 / 习惯 / 偏好 / 生活方式）
2. **聚合统计** — tag / 活动 / 社交 / 场景分布
3. **按天叙事** — `day_summaries` 跨天稳定性
4. **Clip 下钻** — caption / plan vs observation / 偏好证据链
5. **可选** — 从 checkpoint JSONL 流式重算统计

In [4]:
!pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 9.2 MB/s  0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]


In [6]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly 未安装，将回退到 matplotlib")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "DejaVu Sans", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path("..").resolve()
if not (PROJECT_ROOT / "output").exists():
    PROJECT_ROOT = Path.cwd()
    if (PROJECT_ROOT / "EgoTailor").exists():
        PROJECT_ROOT = PROJECT_ROOT / "EgoTailor"

ANALYSIS_DIR = PROJECT_ROOT / "output" / "analysis" / "full_21d_vlm"
REPORT_PATH = ANALYSIS_DIR / "full_21d_behavior_profile.json"
CHECKPOINT_PATH = ANALYSIS_DIR / "clip_analyses_checkpoint.jsonl"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"REPORT_PATH  : {REPORT_PATH}  exists={REPORT_PATH.exists()}")
print(f"CHECKPOINT   : {CHECKPOINT_PATH}  exists={CHECKPOINT_PATH.exists()}")
print(f"plotly       : {HAS_PLOTLY}")

PROJECT_ROOT : /Users/wenwang/Documents/EgoTailor
REPORT_PATH  : /Users/wenwang/Documents/EgoTailor/output/analysis/full_21d_vlm/full_21d_behavior_profile.json  exists=True
CHECKPOINT   : /Users/wenwang/Documents/EgoTailor/output/analysis/full_21d_vlm/clip_analyses_checkpoint.jsonl  exists=True
plotly       : True


## 1. 加载报告

In [7]:
with open(REPORT_PATH) as f:
    report = json.load(f)

meta = report["metadata"]
stats = report["aggregate_stats"]
day_summaries = report["day_summaries"]
profile = report["behavior_profile"] or {}
clips = report["clip_analyses"]

print("=== Metadata ===")
for k in ["created_at", "model", "frames_per_clip", "n_clips_requested", "n_clips_analyzed", "elapsed_sec"]:
    print(f"  {k}: {meta.get(k)}")
print(f"\nTop-level keys: {list(report.keys())}")
print(f"Days: {len(day_summaries)} | Clips: {len(clips)}")
print(f"Status: {stats['status_counts']} | vision_used={stats['vision_used']}")

=== Metadata ===
  created_at: 2026-07-15T18:29:40.516491+00:00
  model: Qwen3-VL-8B-Instruct
  frames_per_clip: 64
  n_clips_requested: 356
  n_clips_analyzed: 356
  elapsed_sec: 33903.9

Top-level keys: ['metadata', 'aggregate_stats', 'day_summaries', 'behavior_profile', 'clip_analyses']
Days: 21 | Clips: 356
Status: {'ok': 356, 'no_video': 0, 'error': 0} | vision_used=356


## 2. 全局行为画像 (`behavior_profile`)

这是 VLM 基于 21 天 day summaries + 聚合统计合成的人设结论。

In [8]:
display(Markdown(f"### Summary\n\n{profile.get('summary', '_(missing)_')}\n"))
display(Markdown(f"### Weekday vs Weekend\n\n{profile.get('weekday_vs_weekend', '_(missing)_')}\n"))
display(Markdown(f"### Data quality\n\n{profile.get('data_quality_note', '_(missing)_')}\n"))

traits = profile.get("lifestyle_traits") or []
print("Lifestyle traits:", ", ".join(traits) if traits else "(none)")

### Summary

This individual leads a structured, routine-driven life centered around domestic comfort, quiet solitude, and hands-on creative or mechanical tasks. They prioritize cleanliness, organization, and personal comfort in their environment, often beginning days with hygiene and chores before transitioning into cooking, crafting, or light socializing. Their behavior shows adaptability to weather and schedule changes, favoring indoor activities during rain or snow, yet maintaining a consistent preference for early-morning routines and physical activity like baseball. They enjoy aesthetic environments, creative hobbies, and solitary leisure, with occasional low-key social interactions that feel natural rather than forced.


### Weekday vs Weekend

Weekdays are structured around hygiene, chores, and routine tasks, often transitioning into work or recreational activities like baseball. Weekends show more flexibility — e.g., shopping trips, creative projects, or leisurely walks — but still maintain the same preference for quiet, organized, and hands-on activities. No explicit weekend data is provided, but the pattern suggests continuity with weekday habits.


### Data quality

Full coverage (356 clips, 100% vision used), no errors or missing video. Captions are reliable for behavioral inference, with strong evidence from day summaries and activity tags. Some preferences are inferred from context (e.g., 'prefers quiet, solitary activities') rather than direct tags, but supported by consistent patterns across multiple days.


Lifestyle traits: disciplined, organized, solitary, hands-on, adaptable, comfort-seeking, creative


In [9]:
interests_df = pd.DataFrame(profile.get("core_interests") or [])
habits_df = pd.DataFrame(profile.get("habitual_patterns") or [])
prefs_df = pd.DataFrame(profile.get("preferences") or [])

print("=== Core interests ===")
if not interests_df.empty:
    display(interests_df[["topic", "confidence", "evidence"]].style.set_properties(**{"text-align": "left"}))
else:
    print("(empty)")

print("\n=== Habitual patterns ===")
if not habits_df.empty:
    display(habits_df)
else:
    print("(empty)")

print("\n=== Preferences ===")
if not prefs_df.empty:
    display(prefs_df)
else:
    print("(empty)")

=== Core interests ===


,topic,confidence,evidence
0,domestic craftsmanship and maintenance,high,"Frequent activities include sewing, painting, repairing, and organizing household items, often in a garage or kitchen setting. They also engage in 'routine maintenance' and 'home improvement' with precision and care."
1,cooking and food preparation,high,"Repeatedly seen preparing meals with gloves, using kitchen tools methodically, and cleaning after cooking — often in the morning or evening. They also prefer cooking at home and show attention to ingredient organization."
2,"quiet, solitary leisure",high,"Prefers activities like watching TV, playing cards alone, or sewing while listening to music — often in a calm, dimly lit, or aesthetically decorated space. Socializing is minimal and low-key."
3,early-morning routines,high,"Consistently begins days with hygiene, breakfast prep, and chores — often in dim or natural light — suggesting a disciplined, ritualistic approach to mornings."



=== Habitual patterns ===


,pattern,time_context,frequency_hint
0,starts day with hygiene and household chores,weekday morning,"daily, consistent across all weekdays"
1,prepares breakfast in kitchen with gloves,weekday morning,"daily, often with methodical care"
2,engages in creative or mechanical work in gara...,weekday afternoon or evening,"weekly, often after routine tasks"
3,prefers indoor activities during rain or snow,weekdays during weather anomalies,"occasional, but consistent in behavior"
4,drives to recreational or social destinations,"weekdays, often early morning","occasional, but recurring when weather permits"



=== Preferences ===


,category,preference,reasoning
0,home,"prefers organized, tidy, and aesthetically ple...",Frequent actions include arranging kitchen ite...
1,leisure,"enjoys quiet, solitary, hands-on activities li...",Activities like sewing a floor pillow while wa...
2,food,prefers cooking at home with methodical care a...,"Wearing gloves while cooking, organizing ingre..."
3,social,"enjoys minimal, low-key social interactions",Interactions are brief and non-intrusive — oft...
4,mobility,"prefers driving as a mode of transportation, e...",Driving is frequently observed — including thr...


## 3. 聚合统计可视化

用 clip 级计数核对画像是否被少数噪声带偏。

In [ ]:
def pairs_to_df(pairs, col_a="name", col_b="count"):
    return pd.DataFrame(pairs, columns=[col_a, col_b])


tags_df = pairs_to_df(stats.get("top_behavior_tags") or [], "tag", "count")
acts_df = pairs_to_df(stats.get("top_activities") or [], "activity", "count")
pref_topics_df = pairs_to_df(stats.get("top_preference_topics") or [], "topic", "count")
social_df = pd.Series(stats.get("social_interaction_dist") or {}).rename("count").rename_axis("social").reset_index()
scene_df = pd.Series(stats.get("scene_dist") or {}).rename("count").rename_axis("scene").reset_index()

display(Markdown("**Top behavior tags**"))
display(tags_df.head(15))
display(Markdown("**Top activities**"))
display(acts_df.head(15))
display(Markdown("**Social / Scene**"))
display(pd.concat([social_df, scene_df], axis=1))

In [ ]:
def barh(df, name_col, count_col, title, top_n=15, figsize=(8, 5)):
    plot_df = df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(plot_df[name_col].astype(str), plot_df[count_col])
    ax.set_title(title)
    ax.set_xlabel("count")
    fig.tight_layout()
    plt.show()


if HAS_PLOTLY:
    display(px.bar(tags_df.head(15), x="count", y="tag", orientation="h", title="Top behavior tags").update_layout(yaxis={"categoryorder": "total ascending"}))
    display(px.bar(acts_df.head(15), x="count", y="activity", orientation="h", title="Top activities").update_layout(yaxis={"categoryorder": "total ascending"}))
    display(px.pie(social_df, names="social", values="count", title="Social interaction"))
    display(px.pie(scene_df, names="scene", values="count", title="Scene distribution"))
else:
    barh(tags_df, "tag", "count", "Top behavior tags")
    barh(acts_df, "activity", "count", "Top activities")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].pie(social_df["count"], labels=social_df["social"], autopct="%1.0f%%")
    axes[0].set_title("Social interaction")
    axes[1].pie(scene_df["count"], labels=scene_df["scene"], autopct="%1.0f%%")
    axes[1].set_title("Scene")
    fig.tight_layout()
    plt.show()

## 4. 按天叙事 (`day_summaries`)

In [ ]:
day_rows = []
for key, info in day_summaries.items():
    syn = info.get("synthesis") or {}
    day_rows.append(
        {
            "day_index": int(info.get("day_index", key)),
            "calendar_date": info.get("calendar_date"),
            "day_theme": info.get("day_theme"),
            "n_clips": info.get("n_clips"),
            "day_summary": syn.get("day_summary", ""),
            "dominant_activities": syn.get("dominant_activities") or [],
            "n_habit_candidates": len(syn.get("habit_candidates") or []),
            "n_preference_signals": len(syn.get("preference_signals") or []),
            "n_anomalies": len(syn.get("anomalies") or []),
            "scene_mix": syn.get("scene_mix", ""),
        }
    )

days_df = pd.DataFrame(day_rows).sort_values("day_index").reset_index(drop=True)
display(days_df[["day_index", "calendar_date", "day_theme", "n_clips", "n_habit_candidates", "n_preference_signals", "n_anomalies"]])

In [ ]:
# 浏览某一天的完整 synthesis；改 DAY_TO_SHOW 即可
DAY_TO_SHOW = 0

info = day_summaries[str(DAY_TO_SHOW)]
syn = info.get("synthesis") or {}
display(Markdown(
    f"### Day {DAY_TO_SHOW} — {info.get('calendar_date')} ({info.get('day_theme')})\n\n"
    f"**n_clips** = {info.get('n_clips')}\n\n"
    f"{syn.get('day_summary', '')}\n\n"
    f"**Dominant activities:** {', '.join(syn.get('dominant_activities') or [])}\n\n"
    f"**Scene mix:** {syn.get('scene_mix', '')}\n"
))

habit_cands = pd.DataFrame(syn.get("habit_candidates") or [])
pref_sigs = pd.DataFrame(syn.get("preference_signals") or [])
if not habit_cands.empty:
    print("Habit candidates:")
    display(habit_cands)
if not pref_sigs.empty:
    print("Preference signals:")
    display(pref_sigs)
anoms = syn.get("anomalies") or []
print("Anomalies:", anoms if anoms else "(none)")

In [ ]:
# 跨天：哪些活动短语反复出现
act_counter = Counter()
for acts in days_df["dominant_activities"]:
    for a in acts:
        act_counter[str(a).lower()] += 1

cross_day_acts = pd.DataFrame(act_counter.most_common(25), columns=["activity", "n_days"])
display(Markdown("### Activities recurring across days"))
display(cross_day_acts)

if HAS_PLOTLY:
    display(px.bar(cross_day_acts.head(15), x="n_days", y="activity", orientation="h",
                   title="Dominant activities by #days").update_layout(yaxis={"categoryorder": "total ascending"}))
else:
    barh(cross_day_acts, "activity", "n_days", "Dominant activities by #days")

## 5. Clip 级表 & 时间分布

In [ ]:
def flatten_clip(c: dict) -> dict:
    a = c.get("analysis") if isinstance(c.get("analysis"), dict) else {}
    return {
        "video_uid": c.get("video_uid"),
        "day_index": c.get("day_index"),
        "calendar_date": c.get("calendar_date"),
        "day_of_week": c.get("day_of_week"),
        "slot_id": c.get("slot_id"),
        "plan_chunk": c.get("plan_chunk"),
        "main_scene": c.get("main_scene"),
        "day_theme": c.get("day_theme"),
        "start_timestamp": c.get("start_timestamp"),
        "end_timestamp": c.get("end_timestamp"),
        "duration_min": c.get("duration_min"),
        "status": c.get("status"),
        "vision_used": c.get("vision_used"),
        "social_interaction": a.get("social_interaction"),
        "behavior_tags": a.get("behavior_tags") or [],
        "observed_activities": a.get("observed_activities") or [],
        "caption": a.get("caption") or "",
        "plan_vs_observation": a.get("plan_vs_observation") or "",
        "n_pref_hypotheses": len(a.get("preference_hypotheses") or []),
        "n_habit_signals": len(a.get("habit_signals") or []),
        "preference_hypotheses": a.get("preference_hypotheses") or [],
        "habit_signals": a.get("habit_signals") or [],
    }


clip_df = pd.DataFrame([flatten_clip(c) for c in clips])
clip_df["start_timestamp"] = pd.to_datetime(clip_df["start_timestamp"], errors="coerce")
clip_df["hour"] = clip_df["start_timestamp"].dt.hour

print(clip_df.shape)
display(clip_df[["day_index", "calendar_date", "day_of_week", "slot_id", "plan_chunk", "main_scene",
                 "social_interaction", "n_pref_hypotheses", "n_habit_signals"]].head(12))

In [ ]:
by_day = clip_df.groupby("day_index").size().rename("n_clips").reset_index()
by_dow = clip_df.groupby("day_of_week").size().rename("n_clips").reset_index()
by_hour = clip_df.groupby("hour").size().rename("n_clips").reset_index()
by_slot = clip_df["slot_id"].value_counts().head(20).rename_axis("slot_id").reset_index(name="n_clips")

if HAS_PLOTLY:
    display(px.bar(by_day, x="day_index", y="n_clips", title="Clips per day"))
    display(px.bar(by_hour, x="hour", y="n_clips", title="Clips by start hour"))
    display(px.bar(by_slot, x="n_clips", y="slot_id", orientation="h",
                   title="Top slot_id").update_layout(yaxis={"categoryorder": "total ascending"}))
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].bar(by_day["day_index"], by_day["n_clips"])
    axes[0].set_title("Clips per day")
    axes[1].bar(by_hour["hour"], by_hour["n_clips"])
    axes[1].set_title("Clips by hour")
    axes[2].barh(by_slot["slot_id"][::-1], by_slot["n_clips"][::-1])
    axes[2].set_title("Top slot_id")
    fig.tight_layout()
    plt.show()

print("Day-of-week counts:")
display(by_dow)

In [ ]:
# Tag × day heatmap（验证习惯是否跨天稳定）
tag_day = Counter()
all_tag_counts = Counter()
for _, row in clip_df.iterrows():
    for t in row["behavior_tags"]:
        t = str(t).lower()
        tag_day[(t, int(row["day_index"]))] += 1
        all_tag_counts[t] += 1

top_tags = [t for t, _ in all_tag_counts.most_common(12)]

mat = np.zeros((len(top_tags), 21), dtype=int)
for i, tag in enumerate(top_tags):
    for d in range(21):
        mat[i, d] = tag_day.get((tag, d), 0)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(mat, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(len(top_tags)))
ax.set_yticklabels(top_tags)
ax.set_xticks(range(21))
ax.set_xlabel("day_index")
ax.set_title("Behavior tag frequency by day")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()

## 6. 证据下钻：caption / 偏好 / plan vs observation

从画像结论回查原始 caption，确认证据是否扎实。

In [ ]:
def show_clip(uid_or_row):
    """传入 video_uid 或 clip_df 的一行。"""
    if isinstance(uid_or_row, str):
        rows = clip_df[clip_df["video_uid"] == uid_or_row]
        if rows.empty:
            print("not found:", uid_or_row)
            return
        row = rows.iloc[0]
    else:
        row = uid_or_row

    display(Markdown(
        f"### `{row['video_uid']}` — day {row['day_index']} / {row['calendar_date']} / `{row['slot_id']}`\n\n"
        f"**Plan:** {row['plan_chunk']}\n\n"
        f"**Scene / social:** {row['main_scene']} / {row['social_interaction']}\n\n"
        f"**Tags:** {', '.join(row['behavior_tags'])}\n\n"
        f"**Caption**\n\n{row['caption']}\n\n"
        f"**Plan vs observation**\n\n{row['plan_vs_observation']}\n"
    ))
    if row["preference_hypotheses"]:
        print("Preference hypotheses:")
        display(pd.DataFrame(row["preference_hypotheses"]))
    if row["habit_signals"]:
        print("Habit signals:", row["habit_signals"])


# 默认看 day0 前 2 个 clip；也可改成任意 uid
for _, r in clip_df[clip_df["day_index"] == 0].head(2).iterrows():
    show_clip(r)

In [ ]:
# 按关键词检索 caption / plan（用于验证画像中的某个 claim）
QUERY = "cook"  # 试试: sew / garage / commute / shop / exercise / hygiene

q = QUERY.lower()
mask = (
    clip_df["caption"].str.lower().str.contains(q, na=False)
    | clip_df["plan_chunk"].astype(str).str.lower().str.contains(q, na=False)
    | clip_df["behavior_tags"].apply(lambda xs: any(q in str(t).lower() for t in xs))
)
hits = clip_df.loc[mask, ["day_index", "calendar_date", "slot_id", "plan_chunk", "behavior_tags", "video_uid"]]
print(f"QUERY={QUERY!r} → {len(hits)} clips")
display(hits.head(20))

if len(hits):
    show_clip(hits.iloc[0]["video_uid"])

In [ ]:
# 汇总所有 preference hypotheses（按 topic 聚合）
pref_rows = []
for _, row in clip_df.iterrows():
    for ph in row["preference_hypotheses"]:
        if not isinstance(ph, dict):
            continue
        pref_rows.append({
            "topic": str(ph.get("topic", "")).lower(),
            "confidence": ph.get("confidence"),
            "evidence": ph.get("evidence"),
            "day_index": row["day_index"],
            "video_uid": row["video_uid"],
            "slot_id": row["slot_id"],
        })

pref_all = pd.DataFrame(pref_rows)
if pref_all.empty:
    print("No preference hypotheses")
else:
    topic_counts = (
        pref_all.groupby("topic")
        .agg(n=("video_uid", "count"), n_days=("day_index", "nunique"),
             high_conf=("confidence", lambda s: (s == "high").sum()))
        .sort_values("n", ascending=False)
        .reset_index()
    )
    display(Markdown("### Preference topics across clips"))
    display(topic_counts.head(25))

    TOPIC = topic_counts.iloc[0]["topic"]
    print(f"\nEvidence samples for top topic: {TOPIC!r}")
    display(pref_all[pref_all["topic"] == TOPIC][["day_index", "slot_id", "confidence", "evidence", "video_uid"]].head(8))

## 7. Plan vs Observation 对齐抽查

粗略用关键词启发式标出「看起来对齐 / 偏移」的 clip（仅辅助浏览，非严格评测）。

In [ ]:
ALIGN_WORDS = ("align", "match", "consistent", "correspond", "follows", "as planned", "matches the")
DRIFT_WORDS = ("differ", "unlike", "instead", "not match", "does not", "doesn't", "deviation", "unexpected")


def alignment_label(text: str) -> str:
    t = (text or "").lower()
    a = sum(w in t for w in ALIGN_WORDS)
    d = sum(w in t for w in DRIFT_WORDS)
    if a and not d:
        return "likely_aligned"
    if d and not a:
        return "likely_drift"
    if a and d:
        return "mixed"
    return "unclear"


clip_df["plan_align"] = clip_df["plan_vs_observation"].map(alignment_label)
align_counts = clip_df["plan_align"].value_counts().rename_axis("label").reset_index(name="n")
display(align_counts)

print("\nSample likely_drift clips:")
drift = clip_df[clip_df["plan_align"] == "likely_drift"][
    ["day_index", "slot_id", "plan_chunk", "plan_vs_observation", "video_uid"]
]
display(drift.head(10))
if len(drift):
    show_clip(drift.iloc[0]["video_uid"])

## 8. （可选）从 checkpoint JSONL 流式重算

当完整 JSON 太大、或只想验证 `aggregate_stats` 是否可复现时用。

In [ ]:
def aggregate_from_jsonl(path: Path) -> dict:
    tag_c, act_c, social_c, scene_c, pref_c = Counter(), Counter(), Counter(), Counter(), Counter()
    status_c = Counter()
    vision = 0
    n = 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            n += 1
            status_c[r.get("status") or "unknown"] += 1
            vision += int(bool(r.get("vision_used")))
            scene_c[str(r.get("main_scene") or "unknown")] += 1
            a = r.get("analysis") or {}
            if not isinstance(a, dict):
                continue
            for t in a.get("behavior_tags") or []:
                tag_c[str(t).lower()] += 1
            for act in a.get("observed_activities") or []:
                act_c[str(act).lower()] += 1
            if a.get("social_interaction"):
                social_c[str(a["social_interaction"]).lower()] += 1
            for ph in a.get("preference_hypotheses") or []:
                if isinstance(ph, dict) and ph.get("topic"):
                    pref_c[str(ph["topic"]).lower()] += 1
    return {
        "n_clips": n,
        "status_counts": dict(status_c),
        "vision_used": vision,
        "top_behavior_tags": tag_c.most_common(10),
        "top_activities": act_c.most_common(10),
        "social_interaction_dist": dict(social_c),
        "scene_dist": dict(scene_c),
        "top_preference_topics": pref_c.most_common(10),
    }


if CHECKPOINT_PATH.exists():
    recomputed = aggregate_from_jsonl(CHECKPOINT_PATH)
    print("From JSONL:", {k: recomputed[k] for k in ["n_clips", "status_counts", "vision_used"]})
    print("Report    :", {k: stats[k] for k in ["n_clips", "status_counts", "vision_used"]})
    print("\nJSONL top tags:", recomputed["top_behavior_tags"][:5])
    print("Report top tags:", stats["top_behavior_tags"][:5])
else:
    print("checkpoint missing:", CHECKPOINT_PATH)

## 9. 一页摘要导出

把关键结论打成 Markdown，方便贴进笔记 / PR。

In [ ]:
lines = [
    f"# EgoTailor 21-day behavior profile",
    f"",
    f"- Model: `{meta.get('model')}` | clips: {meta.get('n_clips_analyzed')} | frames/clip: {meta.get('frames_per_clip')}",
    f"- Status: `{stats.get('status_counts')}` | vision_used={stats.get('vision_used')}",
    f"",
    f"## Summary",
    profile.get("summary", ""),
    f"",
    f"## Lifestyle traits",
    ", ".join(profile.get("lifestyle_traits") or []),
    f"",
    f"## Core interests",
]
for x in profile.get("core_interests") or []:
    lines.append(f"- **{x.get('topic')}** ({x.get('confidence')}): {x.get('evidence')}")

lines += ["", "## Habitual patterns"]
for x in profile.get("habitual_patterns") or []:
    lines.append(f"- {x.get('pattern')} — {x.get('time_context')} ({x.get('frequency_hint')})")

lines += ["", "## Preferences"]
for x in profile.get("preferences") or []:
    lines.append(f"- **{x.get('category')}**: {x.get('preference')}")

lines += [
    "",
    "## Top tags",
    ", ".join(f"{t}({c})" for t, c in (stats.get("top_behavior_tags") or [])[:10]),
    "",
    "## Weekday vs weekend",
    profile.get("weekday_vs_weekend", ""),
]

summary_md = "\n".join(lines)
out_md = ANALYSIS_DIR / "behavior_profile_summary.md"
out_md.write_text(summary_md, encoding="utf-8")
print(f"Wrote {out_md}")
display(Markdown(summary_md))